<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_judgments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [3]:
# @title Dependencies

# Installations
!pip install -q pyarrow
!pip install -q tqdm

# First-party imports
from google.colab import drive
import google.colab.userdata

# Standard library imports
import random
import json
import os
import asyncio
import math

# Third-party imports
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import login, InferenceClient
import tqdm.asyncio

In [2]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews_processed")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the llm reviews (and their logs)
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews_judged")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_evaluated.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_evaluated.jsonl")

# Define the full log path (final Parquet output)
LOG_PATH = os.path.join(RAW_DIR, "llm_reviews_log.parquet")
# Define the full log path for intermediate JSONL storage
LOG_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_log.jsonl")

Mounted at /content/drive
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_processed.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged.


In [20]:
# @title Setup definitions

### --- Settings --- ###

SIMULATION_ACTIVE = True # SIMULATION_ACTIVE = False activates the inference endpoint call
MAX_CONCURRENT_CALLS = 5

# Subsetting dataset content
SUBSETTING_ACTIVE = True # SUBSETTING_ACTIVE = True activates the data subsetting
NUM_SUBSET = 2

### --- API/Client Settings --- ###
MAX_RETRIES = 3
RETRY_DELAY = 3

# Error messages for client
JUDGE_GENERATION_FAILURE = "Error: Failed to generate judgement."
UNKNOWN_ERROR_STATE = "Error: Unknown client error."

### --- Content --- ####

TASK_DESCRIPTION_HEADER = '''###Task Description: You are an expert in evaluating peer review comments with respect to different aspects. These aspects are aimed to maximize the utilization of the review comments for the authors. The primary purpose of the review is to help/guide authors in improving their drafts. Keep this in mind while evaluating the review point. Whenever you encounter a borderline case, think: “Will this review point help authors improve their draft?”. There is no correlation between the aspect score and the length of the review point.'''

ASPECTS_NO_EXAMPLES = {
"actionability":
'''
**Actionability**

**Definition:** Measures the level of actionability in a review point. We evaluate actionability based on two criteria:

1. **Explicit vs. Implicit**:
   - **Explicit:** Actions or suggestions that are direct or apparent. Authors can directly identify modifications they should apply to their draft. Clarification questions should be treated as explicit statements if they give a direct action.
   - **Implicit:** Actions that need to be inferred from the comment. This includes missing parts that need to be added. Authors can deduce what needs to be done after reading the comment.

2. **Concrete vs. Vague**:
   - **Concrete:** Once the action is identified, the authors know exactly what needs to be done and how to apply the action.
   - **Vague:** After identifying the action, the authors still don’t know how to carry out this action.

**Importance:** It’s more important for actions to be concrete so that authors know how to apply them. It's also preferred for actions to be stated directly rather than inferred.

**Actionability Scale (1-5):**

1. **1: Unactionable**
   - **Definition:** The comment lacks meaningful information to help authors improve the paper. Authors do not know what they should do after reading the comment.

2. **2: Borderline Actionable**
   - **Definition:** The comment includes an implicitly stated action or an action that can be inferred. However, the action itself is vague and lacks detail on how to apply it.

3. **3: Somewhat Actionable**
   - **Definition:** The comment explicitly states an action but is vague on how to execute it.

4. **4: Mostly Actionable**
   - **Definition:** The comment implicitly states an action but concretely states how to implement the inferred action.

5. **5: Highly Actionable**
   - **Definition:** The comment contains an explicit action and concrete details on how to implement it. Authors know exactly how to apply it.
''',

"grounding_specificity": '''

**Grounding Specificity**

**Definition:** Measures how explicitly a review comment refers to a specific part of the paper and how clearly it identifies the issue with that part. This helps authors understand what needs revision and why. Grounding specificity has two key components:

1. **Grounding:** How well the authors can identify the specific part of the paper being addressed.
   - **Weak Grounding:** The author can make an educated guess but cannot precisely identify the referenced part.
   - **Full Grounding:** The author can accurately pinpoint the section, table, figure, or unique aspect being addressed. This can be achieved through:
     - Literal mentions of sections, tables, figures, etc.
     - Mentions of unique elements of the paper.
     - General comments that clearly imply the relevant parts without explicitly naming them.

2. **Specificity:** How clearly the comment details what is wrong or missing in the referenced part. If external work is mentioned, it also evaluates whether specific examples are provided.

**Importance:** It's more important for the comment to be grounded than to be specific.

**Grounding Specificity Scale (1-5):**

1. **Not Grounded**
   - **Definition**: This comment is not grounded at all. It does not identify a specific area in the paper. The comment is highly unspecific.

2. **Weakly Grounded and Not Specific**
   - **Definition**: The authors cannot confidently determine which part the comment addresses. Further, the comment does not specify what needs to be addressed in this part.

3. **Weakly Grounded and Specific**
   - **Definition**: The authors cannot confidently determine which part the comment addresses. However, the comment clearly specifies what needs to be addressed in this part.

4. **Fully Grounded and Under-Specific**
   - **Definition**: The comment explicitly mentions which part of the paper it addresses, or it should be obvious to the authors. However, this comment does not specify what needs to be addressed in this part.

5. **Fully Grounded and Specific**
   - **Definition**: The comment explicitly mentions which part of the paper it addresses, and it is obvious to the authors. The comment specifies what needs to be addressed in this part.
''',

"verifiability":
'''
**Verifiability**

**Definition:** Evaluates whether a review comment contains a claim and, if so, how well that claim is supported using logical reasoning, common knowledge, or external references.

### **Step 1: Claim Extraction**

**Objective:**
Determine whether the given text contains a claim (i.e., an opinion, judgment, or suggestion) or consists solely of factual statements that require no verification.

**Claim Definition:**
A statement is considered a claim if it falls into one or more of the following categories:
- **Subjective opinions or disagreements** (e.g., criticism of an experimental choice).
- **Suggestions or requests for changes** (e.g., recommending removal, addition, or discussion).
- **Judgments about the paper** (e.g., stating something is unclear, not well-written, or lacks detail).
- **Deductions or inferred observations** that go beyond merely stating facts.
- **Statements requiring justification** to be understood or accepted.


**Normal Statements ("No Claim")**
A statement is classified as "X" if it:
- Describes facts without suggesting changes.
- Makes general statements about the paper without an opinion.
- Presents objective, verifiable facts that require no justification.
- Asks for clarifications or general questions.
- States logical statements or directly inferable information.
- Makes positive claims (e.g., *“The paper is well-written”*), as these do not help improve the work.

---

### **Step 2: Verifiability Verification**

**Objective:**
Assess how well a claim is verified by examining the reasoning, common knowledge, or external references provided. The purpose is to ensure that the review comment helps the authors improve their work.

**Verification Methods:**
A claim is considered verifiable if supported by one or more of the following:
- **Logical reasoning** – A clear explanation of why the claim is valid.
- **Common knowledge** – Reference to well-accepted practices or standards.
- **External references** – Citation of relevant literature, data, or sources.

**Verifiability Scale (1–5 & X):**

1. **1: Unverifiable**
   - The comment contains a claim without any supporting evidence or justification.
2. **2: Borderline Verifiable**
   - Some support is provided, but it is vague, insufficient, or difficult to follow.
3. **3: Somewhat Verifiable**
   - The claim has some justification but lacks key elements (e.g., examples, references).
4. **4: Mostly Verifiable**
   - The claim is well-supported but has minor gaps in explanation or references.
5. **5: Fully Verifiable**
   - The claim is thoroughly supported by explicit, sufficient, and robust evidence, such as:
     - Clear reasoning and precise explanations.
     - Specific references to external works.
     - Logical and unassailable common-sense arguments.
6. **X: No Claim**
- The comment contains only factual, descriptive statements without claims, opinions, or suggestions.

---

### **Instructions for Evaluation:**
1. **Extract Claims:** Determine whether the review comment contains a claim or is a normal statement. If there is no claim, score it as "X"
2. **Assess Verifiability:** If a claim exists, score it based on how well it is justified from 1 to 5.
'''
,

"helpfulness" : '''
**Helpfulness**

**Definition:** Assign a subjective score to reflect the value of the review comment to the authors. Helpfulness is rated on a scale from 1 to 5, with the following definitions:

1. **1: Not Helpful at All**
   - **Definition:** The comment fails to identify meaningful weaknesses or suggest improvements, leaving the authors with no actionable feedback.

2. **2: Barely Helpful**
   - **Definition:** The comment identifies a weakness or improvement area but is vague, lacks clarity, or provides minimal guidance, making it only slightly beneficial for the authors.

3. **3: Somewhat Helpful**
   - **Definition:** The comment identifies weaknesses or areas for improvement but is incomplete or lacks depth. While the authors gain some insights, the feedback does not fully address their needs for improving the draft.

4. **4: Mostly Helpful**
   - **Definition:** The comment provides clear and actionable feedback on weaknesses and areas for improvement, though it could be expanded or refined to be fully comprehensive and impactful.

5. **5: Highly Helpful**
   - **Definition:** The comment thoroughly identifies weaknesses and offers detailed, actionable, and constructive suggestions that empower the authors to significantly improve their draft.
'''
}

INSTRUCTION_SCORE_ONLY_PROMPT_TAIL = '''
###Instruction:
Evaluate the review based on the given definitions of the aspect(s) above. Output only the score. The possbile values for scores are 1-5 and X.

Generate the output in JSON format with the following format:

   "actionability_label": "",
   "grounding_specificity_label": "",
   "verifiability_label": "",
   "helpfulness_label": ""


###Review Point:
{review_point}''' ### According to the repo code, but with deviation from the examplary prompt in the paper

### --- API/Client --- ###

# HF token key
hf_token = google.colab.userdata.get('HUGGINGFACE_TOKEN')
login(token=hf_token)

# HF Inference endpoint
HF_INFERENCE_ENDPOINT_URL = "boda/RevUtil_Llama-3.1-8B-Instruct_score_only" # fine-tuned score-only model

In [14]:
# @title Data import

# Read processed llm reviews
df_reviews = pd.read_parquet(os.path.join(DATASET_DIR, "llm_reviews_segmented.parquet"))

# Check df_reviews
display(df_reviews.head(3))
display(df_reviews.shape)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes_1,llm_review_1,llm_notes_2,llm_review_2,extracted_weakness,segmented_comment
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,Weaknesses\nStrengths\n- Clear problem identif...,"- Clear problem identification: the ""graph mis..."
1,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,Weaknesses\nStrengths\n- Clear problem identif...,"- Simple, scalable method: joint generator/dis..."
2,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,Weaknesses\nStrengths\n- Clear problem identif...,- Strong empirical gains: consistent improveme...


(218, 13)

In [15]:
# @title Subsetting (if active)

if SUBSETTING_ACTIVE:
    df_reviews = df_reviews.head(NUM_SUBSET)
    print(f"Data subsetted to {NUM_SUBSET} rows.")
    display(df_reviews.head(3))
    display(df_reviews.shape)
else:
    print("Paper content subsetting is inactive.")

Data subsetted to 2 rows.


,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes_1,llm_review_1,llm_notes_2,llm_review_2,extracted_weakness,segmented_comment
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,Weaknesses\nStrengths\n- Clear problem identif...,"- Clear problem identification: the ""graph mis..."
1,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,Weaknesses\nStrengths\n- Clear problem identif...,"- Simple, scalable method: joint generator/dis..."


(2, 13)

## Functions

In [21]:
# @title Prompt construction

def build_prompt(review_point: str) -> list[dict]:
    """Constructs the full prompt in messages format for chat_completion."""
    system_content_parts = []
    system_content_parts.append(TASK_DESCRIPTION_HEADER)

    for aspect_key, aspect_definition_text in ASPECTS_NO_EXAMPLES.items():
        if aspect_key == "grounding_specificity":
            display_aspect_name = "Grounding & Specificity"
        else:
            display_aspect_name = aspect_key.replace('_', ' ').title()
        system_content_parts.append(f"Aspect: {display_aspect_name} {aspect_definition_text}")

    instruction_part = INSTRUCTION_SCORE_ONLY_PROMPT_TAIL.split("###Review Point:")[0].strip()
    system_content_parts.append(instruction_part)

    system_message_content = "\n".join(system_content_parts)

    user_message_content = f"###Review Point:\n{review_point}"

    messages = [
        {"role": "system", "content": system_message_content},
        {"role": "user", "content": user_message_content} ### Originally the authors did not displayed any difference between system and user messsage
    ]
    return messages

In [17]:
# @title Pydantic Score Schema

from pydantic import BaseModel, ValidationError, Field
from typing import Literal

class ReviewScore(BaseModel):
    actionability_label: Literal["1", "2", "3", "4", "5", "X"]
    grounding_specificity_label: Literal["1", "2", "3", "4", "5", "X"]
    verifiability_label: Literal["1", "2", "3", "4", "5", "X"]
    helpfulness_label: Literal["1", "2", "3", "4", "5", "X"]


In [22]:
# @title HuggingFaceScoringClient

class HuggingFaceScoringClient:
    def __init__(self, endpoint_url: str, hf_token: str, simulate: bool = False, seed: int = 42):
        """
        Initialize HuggingFaceScoringClient for a fixed HuggingFace Inference Endpoint.
        This client handles both simulation and actual API calls, including retry logic.
        """
        self.simulate = simulate
        self.rng = random.Random(seed)
        self.endpoint_url = endpoint_url
        self.hf_token = hf_token

        if not self.simulate:
            if not self.endpoint_url or self.endpoint_url == "YOUR_HUGGINGFACE_INFERENCE_ENDPOINT_URL_HERE":
                raise ValueError("HF_INFERENCE_ENDPOINT_URL must be set when simulation is inactive.")
            self._hf_client = InferenceClient(model=self.endpoint_url, token=self.hf_token)
            print(f"HuggingFaceScoringClient initialized for endpoint: {self.endpoint_url}")
        else:
            print("HuggingFaceScoringClient running in simulation mode.")

    async def score_review_point(self, review_point: str) -> str:
        """
        Scores a single review point by building the prompt and calling the HF Inference API
        or returning a simulated response. Handles retry logic internally.
        """
        messages = build_prompt(review_point)

        if self.simulate:
            await asyncio.sleep(0.1)
            simulated_output = {
                "actionability_label": str(self.rng.randint(1, 5)),
                "grounding_specificity_label": str(self.rng.randint(1, 5)),
                "verifiability_label": self.rng.choice([str(self.rng.randint(1, 5)), "X"]),
                "helpfulness_label": str(self.rng.randint(1, 5))
            }
            return json.dumps(simulated_output)

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = await asyncio.to_thread(
                    self._hf_client.chat_completion,
                    messages=messages,
                    # grammar={"type": "json", "value": ReviewScore.model_json_schema()}, ### EXCLUDE FOR THE MOMENT TO TEST THE GENERAL WORKFLOW
                )
                json_output = response.choices[0].message.content
                return json_output
            except Exception as e:
                print(f"[LLM ERROR - HuggingFace Client] attempt={attempt} -> {e}", flush=True)
                if attempt == MAX_RETRIES:
                    return json.dumps({"error": JUDGE_GENERATION_FAILURE})
                await asyncio.sleep(RETRY_DELAY)
        return json.dumps({"error": UNKNOWN_ERROR_STATE})

## Execution

In [23]:
# Initialize client
client = HuggingFaceScoringClient(
    endpoint_url=HF_INFERENCE_ENDPOINT_URL,
    hf_token=hf_token,
    simulate=SIMULATION_ACTIVE
)

### INCLUDE KEY LOGIC TO AVOID MULTIPLE SCORING

async def process_review_point(review_id, review_point, semaphore):
    async with semaphore:

        # Score the comment
        raw_json_output = await client.score_review_point(review_point)

        # Add review_id to JSON results
        parsed_output = json.loads(raw_json_output)
        result_entry = {'review_id': review_id, **parsed_output}

        # Prepare log entry
        log_entry = {
            'review_id': review_id,
            'review_point': review_point,
        }
        return result_entry, log_entry

async def main():
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_CALLS)

    total_reviews = len(df_reviews)
    batch_size = MAX_CONCURRENT_CALLS
    num_batches = math.ceil(total_reviews / batch_size)

    # Use tqdm for overall progress
    with tqdm.asyncio.tqdm(total=total_reviews, desc="Overall Scoring Progress") as pbar:
        for i in range(num_batches):
            start_idx = i * batch_size
            end_idx = min((i + 1) * batch_size, total_reviews)
            batch_df = df_reviews.iloc[start_idx:end_idx]

            tasks = []
            for index, row in batch_df.iterrows():
                review_point = row['segmented_comment'] ### ADJUST THE COLUMN NAME IN LATER STAGES
                review_id = row['paper_id'] ### ADJUST THE COLUMN NAME IN LATER STAGES
                tasks.append(process_review_point(review_id, review_point, semaphore))

            # Run tasks for the current batch concurrently
            batch_results = await asyncio.gather(*tasks)

            # Write results and logs for the current batch to JSONL files in append mode
            with open(RESULTS_JSON_PATH, 'a') as results_jsonl_file:
                with open(LOG_JSON_PATH, 'a') as log_jsonl_file:
                    for result_entry, log_entry in batch_results:
                        results_jsonl_file.write(json.dumps(result_entry) + '\n')
                        log_jsonl_file.write(json.dumps(log_entry) + '\n')

            # Update progress bar for the processed batch
            pbar.update(len(batch_results))

    print(f"All review points processed. Results saved to {RESULTS_JSON_PATH} and logs to {LOG_JSON_PATH}.")

# Run the asynchronous main function
await main()


HuggingFaceScoringClient running in simulation mode.


Overall Scoring Progress: 100%|██████████| 2/2 [00:00<00:00, 17.64it/s]

All review points processed. Results saved to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_evaluated.jsonl and logs to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_log.jsonl.


## Results

In [24]:
# @title Convert JSONL to Parquet

# Convert RESULTS_JSON_PATH to RESULTS_PATH (Parquet)
if os.path.exists(RESULTS_JSON_PATH):
    try:
        print(f"Converting {RESULTS_JSON_PATH} to {RESULTS_PATH}...")
        jsonl_df = pd.read_json(RESULTS_JSON_PATH, lines=True)
        if not jsonl_df.empty:
            jsonl_df.to_parquet(RESULTS_PATH, index=False)
            print(f"Successfully converted and saved results to {RESULTS_PATH}.")
        else:
            print(f"{RESULTS_JSON_PATH} is empty. No Parquet file generated for results.")
    except Exception as e:
        print(f"Error converting results JSONL to Parquet: {e}")
else:
    print(f"Intermediate results file '{RESULTS_JSON_PATH}' does not exist. No results Parquet file generated.")

# Convert LOG_JSON_PATH to LOG_PATH (Parquet)
if os.path.exists(LOG_JSON_PATH):
    try:
        print(f"Converting {LOG_JSON_PATH} to {LOG_PATH}...")
        jsonl_log_df = pd.read_json(LOG_JSON_PATH, lines=True)
        if not jsonl_log_df.empty:
            jsonl_log_df.to_parquet(LOG_PATH, index=False)
            print(f"Successfully converted and saved log to {LOG_PATH}.")
        else:
            print(f"'{LOG_JSON_PATH}' is empty. No Parquet file generated for log.")
    except Exception as e:
        print(f"Error converting log JSONL to Parquet: {e}")
else:
    print(f"Intermediate log file '{LOG_JSON_PATH}' does not exist. No log Parquet file generated.")

Converting /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_evaluated.jsonl to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_evaluated.parquet...
Successfully converted and saved results to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_evaluated.parquet.
Converting /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_log.jsonl to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_log.parquet...
Successfully converted and saved log to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_log.parquet.


In [25]:
# @title Read Parquet files

# Load result file
if os.path.exists(RESULTS_PATH):
    try:
        llm_results_df = pd.read_parquet(RESULTS_PATH)
        # Check results only if loading was successful
        display(llm_results_df)
        display(llm_results_df.shape)
    except Exception as e:
        print(f"Warning: Could not read existing results parquet file {RESULTS_PATH}. Error: {e}")
else:
    print(f"Results file '{RESULTS_PATH}' does not exist.")

# Load log file
if os.path.exists(LOG_PATH):
    try:
        llm_log_df = pd.read_parquet(LOG_PATH)
        # Check log only if loading was successful
        display(llm_log_df.head(5))
        display(llm_log_df.shape)
    except Exception as e:
        print(f"Warning: Could not read existing log parquet file {LOG_PATH}. Error: {e}")
else:
    print(f"Log file '{LOG_PATH}' does not exist.")

,review_id,error,actionability_label,grounding_specificity_label,verifiability_label,helpfulness_label
0,author_2,Error: Failed to generate judgement.,NaN,NaN,NaN,NaN
1,author_2,Error: Failed to generate judgement.,NaN,NaN,NaN,NaN
2,author_2,Error: Failed to generate judgement.,NaN,NaN,NaN,NaN
3,author_2,Error: Failed to generate judgement.,NaN,NaN,NaN,NaN
4,author_2,None,1.0,1.0,3.0,2.0
5,author_2,None,2.0,1.0,5.0,5.0


(6, 6)

,review_id,review_point
0,author_2,"- Clear problem identification: the ""graph mis..."
1,author_2,"- Simple, scalable method: joint generator/dis..."
2,author_2,"- Clear problem identification: the ""graph mis..."
3,author_2,"- Simple, scalable method: joint generator/dis..."
4,author_2,"- Clear problem identification: the ""graph mis..."


(6, 2)